# Thesis-scope Colab run (deadline 30 Nov)

PRIMARY: PubMedQA 300 cases x H0-H5,SYC.
SUPPORT: MedQA + MedMCQA 60 cases x H1,H4,H5,SYC.
Small model team, T4 GPU, no API key. Results saved on Google Drive so `--resume` survives disconnects.

**Runtime -> Change runtime type -> T4 GPU.** Push the latest repo from your laptop before starting.

In [ ]:
import os
REPO = "/content/Multi-Agent-medical-diagnosis-llm"
if not os.path.exists(os.path.join(REPO, ".git")):
    !git clone -q https://github.com/HanjalaShihab/Multi-Agent-medical-diagnosis-llm.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}/pilot
print(os.getcwd())

In [ ]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
!pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps

In [ ]:
!mkdir -p data/cases
!python scripts/download_data.py all --out data/raw
!python -m bts_audit.cli prepare --dataset pubmedqa   --raw data/raw/pubmedqa   --out data/cases/pubmedqa_300.jsonl   --n 300
!python -m bts_audit.cli prepare --dataset medqa      --raw data/raw/medqa      --out data/cases/medqa_300.jsonl      --n 300
!python -m bts_audit.cli prepare --dataset medmcqa    --raw data/raw/medmcqa    --out data/cases/medmcqa_300.jsonl    --n 300

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_ROOT = "/content/drive/MyDrive/bts_audit_colab_results"
!mkdir -p {OUT_ROOT}
print(OUT_ROOT)

## Run the thesis scope
If the session disconnects, re-run the setup cells above and **this same cell**.

In [ ]:
!OUT_ROOT='/content/drive/MyDrive/bts_audit_colab_results' bash scripts/run_colab_thesis_scope.sh

In [ ]:
from pathlib import Path
import pandas as pd
root = Path('/content/drive/MyDrive/bts_audit_colab_results')
for d in sorted(root.glob('colab_*')):
    print('\n===', d.name, '===')
    cm = d / 'cell_metrics.csv'
    if not cm.exists():
        print('no cell_metrics.csv yet')
        continue
    df = pd.read_csv(cm)
    cols = [c for c in ['cell','n_cases','accuracy','selection_accuracy_majority','selection_accuracy_confidence','selection_accuracy_bts','selection_accuracy_proper','auc_score','auc_confidence','delta_auc','agreement_rate','rho','delta_auc_oracle'] if c in df.columns]
    print(df[cols].to_string(index=False))

In [ ]:
!zip -rq /content/colab_thesis_results.zip /content/drive/MyDrive/bts_audit_colab_results
from google.colab import files
files.download('/content/colab_thesis_results.zip')